## **Learning Objectives**

By completing these exercises, you will:

- Understand Retrieval-Augmented Generation (RAG) and its components.
- Load, preprocess, and handle PDF documents effectively.
- Convert textual data into embeddings for efficient retrieval.
- Implement and test document retrieval systems using LangChain and FAISS.
- Integrate retrieval systems with free Language Models (LLMs) from ChatGroq .
- Build an interactive chat-based Q&A system.

---

## **Exercise 1: Setup and Warm-up**

In this exercise, you'll set up your environment and select a suitable language model.

**Steps:**

1. **Load Environment Variables:** Ensure your environment variables (e.g., API keys, tokens) are securely stored and loaded.
2. **Choose LLM:** Select a free LLM model from from ChatGroq. 
3. **Instantiate the Model:** Create an instance of your chosen model.


In [1]:
# Import necessary libraries
import os
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEndpoint

# Load environment variables
load_dotenv()

# api_key = os.getenv("GROQ_API_KEY")
# print(api_key)


True

---

## **Exercise 2: Data Ingestion**

In this exercise, you'll learn to load PDF data into a Python environment.

**Steps:**

1. **Import PDF Loader:** Use LangChain’s `PyPDFLoader`.
2. **Load PDF File:** Create a function to read the PDF file.
3. **Display PDF Content:** Print the number of pages and first page content.

In [2]:
# Import PyPDFLoader
from langchain_community.document_loaders import PyPDFLoader

# Example function to load PDF

def load_pdf(pdf_path):
    loader = PyPDFLoader(pdf_path)
    pages = loader.load()
    return pages


In [3]:
# Load your PDF and print out content here
documents = load_pdf("../documents/paracetamol.pdf")

# Eerste pagina printen (optioneel)
print(documents[0].page_content)

202211
178 mm
422 mm
178 mm
422 mm
Front Side Back Side
 Paracetamol 500mg Tablets
178 x 422mm
178 x 30mm
358
202211
NA
Printed Leaﬂet for  Paracetamol 500mg Tablets, Open size: 178 x 422mm, Folding Size : 178x30mm 
Speciﬁcation: 40GSM Bible Paper - Fairmed/Apohilft-Germany 
P4S Complete Solutions
01
Black
Fairmed/Apohilft-Germany 
30mm
Gebrauchsinformation: Information für den Anwender
Paracetamol 500 mg Die Apotheke hilft 
Schmerztabletten
Zur Anwendung bei Kindern ab 4 Jahren, Jugendlichen und Erwachsenen
Lesen Sie die gesamte Packungsbeilage sorgfältig durch, bevor Sie mit der Einnahme dieses Arzneimittels beginnen, denn 
sie enthält wichtige Informationen.
Nehmen Sie dieses Arzneimittel immer genau wie in dieser Packungsbeilage beschrieben bzw. genau nach Anweisung Ihres Arztes 
oder Apothekers ein.
• Heben Sie die Packungsbeilage auf. Vielleicht möchten Sie diese später nochmals lesen.
• Fragen Sie Ihren Apotheker, wenn Sie weitere Informationen oder einen Rat benötigen.
• Wenn S

---

## **Exercise 3: Document Chunking**

This exercise introduces splitting large documents into manageable text chunks.

**Steps:**

1. **Import Text Splitter:** Use `RecursiveCharacterTextSplitter`.
2. **Chunk Document:** Write a function that splits loaded documents into chunks.
3. **Test Function:** Verify by displaying the resulting chunks.


In [4]:
# Import RecursiveCharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Example chunking function
def chunk_documents(documents, chunk_size=200, chunk_overlap=50):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ".", " ", ""]
    )
    
    # Extract raw text from the Document objects
    texts = [doc.page_content for doc in documents]
    full_text = "\n".join(texts)

    # Split into chunks
    chunks = splitter.split_text(full_text)
    return chunks


In [5]:
# Execute your chunking function and display results here

# 1. Chunk the loaded PDF
chunks = chunk_documents(documents)

# 2. Show how many chunks were created
print(f"Number of chunks: {len(chunks)}")

# 3. Display the first chunk
print("\nFirst chunk:\n")
print(chunks[0])


Number of chunks: 129

First chunk:

202211
178 mm
422 mm
178 mm
422 mm
Front Side Back Side
 Paracetamol 500mg Tablets
178 x 422mm
178 x 30mm
358
202211
NA


Technical flow 
# **In één zin**
Embeddings turn text into numerical vectors so the system can compare meaning and retrieve the most relevant chunks.

---

# **What happens technically**

## **1. Chunk → Embedding Model**
Each text chunk is passed into an embedding model, which converts it into a vector.

"Paracetamol dosage info"  
        ↓  
Embedding model  
        ↓  
[0.12, -0.44, 0.87, ...]

---

## **2. Store vectors in FAISS**
FAISS stores all vectors in an index optimized for fast similarity search.

Chunk vectors → FAISS index

---

## **3. User asks a question**
Your question is also converted into a vector.

"What is the max adult dose?"  
        ↓  
Embedding model  
        ↓  
[0.10, -0.40, 0.90, ...]

---

## **4. FAISS finds the closest vectors**
FAISS compares the question vector with all stored vectors and finds the nearest neighbors — the chunks with the most similar meaning.

Question vector → nearest chunk vectors

---

## **5. Relevant chunks go to the LLM**
The retrieved chunks are passed to the LLM as context, enabling it to answer based on the actual PDF content.

Relevant chunks → LLM → Final answer



---

## **Exercise 4: Embedding and Storage**

In this exercise, you will create embeddings from text chunks and store them efficiently.

**Steps:**

1. **Choose Embedding Model:** Use `sentence-transformers/all-mpnet-base-v2` from Hugging Face.
2. **Generate Embeddings:** Transform document chunks into embeddings.
3. **Store Embeddings:** Save these embeddings using FAISS locally.


In [6]:
# Import libraries
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Example function for embeddings and storage
def embed_and_store(chunks):
    # 1. Create embedding model
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

    # 2. Build FAISS vectorstore from text chunks
    vectorstore = FAISS.from_texts(chunks, embedding=embeddings)

    # 3. Save locally
    vectorstore.save_local("faiss_store")

    return vectorstore


In [7]:
# Generate embeddings and save them locally
vectorstore = embed_and_store(chunks)
print("FAISS vectorstore saved successfully.")


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/jopanda/.pyenv/versions/3.11.3/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/jopanda/.pyenv/versions/3.11.3/lib/python3.11/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/Users/jopanda/.pyenv/versions/3.11.3/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 758, in start

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/jopanda/.pyenv/versions/3.11.3/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/jopanda/.pyenv/versions/3.11.3/lib/python3.11/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/Users/jopanda/.pyenv/versions/3.11.3/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 758, in start

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/jopanda/.pyenv/versions/3.11.3/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/jopanda/.pyenv/versions/3.11.3/lib/python3.11/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/Users/jopanda/.pyenv/versions/3.11.3/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 758, in start

AttributeError: _ARRAY_API not found

ImportError: numpy.core._multiarray_umath failed to import

ImportError: Could not import sentence_transformers python package. Please install it with `pip install sentence-transformers`.

---

## **Exercise 5: Retrieval from FAISS**

Here, you will learn how to retrieve documents from a vector database using embeddings.

**Steps:**

1. **Load Embeddings:** Load stored embeddings from the FAISS database.
2. **Implement Retrieval:** Create logic to retrieve relevant chunks based on queries.
3. **Test Retriever:** Execute retrieval using sample queries.

In [ ]:
# Implement retrieval logic from your FAISS database

In [ ]:
# Test your retrieval system with queries

---

## **Exercise 6: Connecting Retrieval with LLM**

You'll now connect document retrieval with the Language Model.

**Steps:**

1. **Create Retrieval Chain:** Link your retrieval system to your instantiated LLM.
2. **Test the Chain:** Confirm it works by generating answers from retrieved documents.

In [ ]:
# Write a function to create retrieval and document processing chains


In [ ]:
# Invoke your chain with a sample question

---

## **Exercise 7: Interactive Chat System**

In the final exercise, build an interactive chat-based query system.

**Steps:**

1. **Create Chat Interface:** Develop a simple function for interactive querying.
2. **Run the Chat:** Allow users to ask questions and receive immediate responses.


In [ ]:
# Define your interactive chat querying function

In [ ]:
# Run and test your interactive chat system

---

## **Conclusion & Reflection**

After completing these exercises:

- Summarize key concepts learned.
- Reflect on the effectiveness and limitations of the free LLM and RAG system you've built.
- Consider how you might improve or extend your system in practical applications.

---